# IMDb India Movie Rating Prediction
### Exploratory Data Analysis, Leakage-Safe Regression & Model Comparison

**Author:** Tayeeb Tahir Buchh

This project predicts the IMDb rating of Indian movies using metadata such as release year, vote count, genre, director, and cast members.

**Target:** `Rating` (a continuous numeric score).

> This is a regression project, not a movie recommendation system. The dataset is not included in this repository. Download `IMDb Movies India.csv` from the Kaggle IMDb India Movies dataset and place it in `data/`, or update `DATA_PATH`.


## 1. Imports and reproducibility


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)


## 2. Load and inspect the dataset


In [ ]:
DATA_PATH = Path("data/IMDb Movies India.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Download the IMDb India Movies CSV "
        "and place it in the data/ folder, or change DATA_PATH."
    )

data = pd.read_csv(DATA_PATH, encoding="ISO-8859-1")
print(f"Dataset shape: {data.shape}")
display(data.head())
display(data.dtypes.to_frame("dtype"))


## 3. Data quality and target preparation

Ratings are the prediction target. Rows without a valid rating cannot be used for supervised regression, so they are removed rather than assigning an artificial rating of zero. The feature preprocessing steps will be fitted inside pipelines after splitting.


In [ ]:
required_columns = ["Rating", "Year", "Votes", "Genre", "Director", "Actor 1", "Actor 2", "Actor 3"]
missing_columns = sorted(set(required_columns) - set(data.columns))
if missing_columns:
    raise ValueError(f"Dataset is missing expected columns: {missing_columns}")

print("Missing values before cleaning:")
display(data[required_columns].isna().sum().to_frame("missing"))

# Parse the target and remove rows without a valid rating.
data["Rating"] = pd.to_numeric(data["Rating"], errors="coerce")
data = data.dropna(subset=["Rating"]).copy()

# Parse year and vote counts without using target information.
data["Year"] = (
    data["Year"].astype(str)
    .str.extract(r"(\d{4})", expand=False)
)
data["Year"] = pd.to_numeric(data["Year"], errors="coerce")

data["Votes"] = (
    data["Votes"].astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
)
data["Votes"] = pd.to_numeric(data["Votes"], errors="coerce")

# Keep the original target scale; report rather than silently clipping unusual values.
print(f"Rows with usable ratings: {len(data):,}")
print(f"Rating range: {data['Rating'].min():.2f} to {data['Rating'].max():.2f}")
display(data["Rating"].describe().to_frame("rating_summary"))


## 4. Exploratory data analysis


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
data["Rating"].hist(bins=30, ax=axes[0])
axes[0].set_title("Distribution of movie ratings")
axes[0].set_xlabel("IMDb rating")
axes[0].set_ylabel("Number of movies")

data["Year"].dropna().hist(bins=30, ax=axes[1])
axes[1].set_title("Distribution of release years")
axes[1].set_xlabel("Release year")
axes[1].set_ylabel("Number of movies")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(data["Votes"], data["Rating"], alpha=0.25)
axes[0].set_title("Votes vs. rating")
axes[0].set_xlabel("Number of votes")
axes[0].set_ylabel("IMDb rating")

axes[1].scatter(data["Year"], data["Rating"], alpha=0.25)
axes[1].set_title("Release year vs. rating")
axes[1].set_xlabel("Release year")
axes[1].set_ylabel("IMDb rating")
plt.tight_layout()
plt.show()


### Most common genres
The genre field may contain multiple comma-separated values. This chart shows the most frequent raw genre strings in the dataset.


In [ ]:
genre_counts = data["Genre"].dropna().astype(str).value_counts().head(15).sort_values()
plt.figure(figsize=(9, 6))
genre_counts.plot(kind="barh")
plt.title("15 most frequent genre entries")
plt.xlabel("Number of movies")
plt.tight_layout()
plt.show()


## 5. Select features and split the data
We use a 70% training, 15% validation, and 15% test split. The test set remains untouched until final evaluation. We deliberately avoid calculating director or actor average ratings using the full dataset: those averages contain information from the target and can leak the answers into the features.


In [ ]:
feature_columns = ["Year", "Votes", "Genre", "Director", "Actor 1", "Actor 2", "Actor 3"]
target_column = "Rating"

X = data[feature_columns].copy()
y = data[target_column].astype(float).copy()

numeric_features = ["Year", "Votes"]
categorical_features = ["Genre", "Director", "Actor 1", "Actor 2", "Actor 3"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE
)

split_summary = pd.DataFrame({
    "rows": [len(y_train), len(y_valid), len(y_test)],
    "mean_rating": [y_train.mean(), y_valid.mean(), y_test.mean()],
}, index=["Train", "Validation", "Test"])
display(split_summary.round(3))


## 6. Build preprocessing and regression pipelines
All imputers, scaling, and category encoding are fitted on the training data only. One-hot encoding handles categories not seen during training by ignoring them. `min_frequency` groups very rare categories to limit the size of the feature matrix.


In [ ]:
numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10,
        max_categories=50,
    )),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocessor, numeric_features),
        ("categorical", categorical_preprocessor, categorical_features),
    ],
    remainder="drop",
)

models = {
    "Ridge Regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", Ridge(alpha=10.0)),
    ]),
    "Random Forest": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(
            n_estimators=250,
            min_samples_leaf=2,
            max_features=0.8,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ]),
}


## 7. Train and compare models on validation data
We report Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R-squared. MAE is the average absolute rating error in rating points. RMSE penalizes larger errors more heavily. R-squared measures variance explained relative to predicting the mean.


In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
    }

validation_rows = []
fitted_models = {}

for name, estimator in models.items():
    estimator.fit(X_train, y_train)
    valid_predictions = estimator.predict(X_valid)
    validation_rows.append({"model": name, **regression_metrics(y_valid, valid_predictions)})
    fitted_models[name] = estimator

validation_results = pd.DataFrame(validation_rows).sort_values("RMSE").reset_index(drop=True)
display(validation_results.round(4))

best_name = validation_results.loc[0, "model"]
print(f"Selected model by lowest validation RMSE: {best_name}")


## 8. Inspect validation predictions
A prediction-versus-actual plot helps reveal systematic under- or over-prediction. The dashed diagonal represents perfect predictions.


In [ ]:
best_model = fitted_models[best_name]
valid_predictions = best_model.predict(X_valid)

plt.figure(figsize=(6, 6))
plt.scatter(y_valid, valid_predictions, alpha=0.3)
limits = [
    min(y_valid.min(), valid_predictions.min()),
    max(y_valid.max(), valid_predictions.max()),
]
plt.plot(limits, limits, linestyle="--")
plt.xlim(limits)
plt.ylim(limits)
plt.xlabel("Actual rating")
plt.ylabel("Predicted rating")
plt.title(f"Validation: actual vs predicted ({best_name})")
plt.tight_layout()
plt.show()


## 9. Refit the selected model and evaluate on the held-out test set
After model selection, refit the selected pipeline on training plus validation data. The test set is used only for the final evaluation.


In [ ]:
X_train_valid = pd.concat([X_train, X_valid], axis=0)
y_train_valid = pd.concat([y_train, y_valid], axis=0)

final_model = models[best_name]
final_model.fit(X_train_valid, y_train_valid)

test_predictions = final_model.predict(X_test)
test_results = pd.DataFrame([regression_metrics(y_test, test_predictions)], index=[best_name])
display(test_results.round(4))

print("Test-set evaluation completed. Do not tune the model using these test results.")


## 10. Test residual analysis
Residual = actual rating − predicted rating. Residual patterns can indicate where the model makes larger or systematic errors.


In [ ]:
residuals = y_test.to_numpy() - test_predictions

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(test_predictions, residuals, alpha=0.3)
axes[0].axhline(0, linestyle="--")
axes[0].set_title("Residuals vs predicted rating")
axes[0].set_xlabel("Predicted rating")
axes[0].set_ylabel("Residual")

axes[1].hist(residuals, bins=30)
axes[1].set_title("Distribution of test residuals")
axes[1].set_xlabel("Actual − predicted rating")
axes[1].set_ylabel("Number of movies")
plt.tight_layout()
plt.show()


## 11. Save the fitted model
The saved pipeline includes preprocessing and the selected regressor. The model artifact is generated locally and should not be committed unless intentionally needed.


In [ ]:
import joblib

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = ARTIFACT_DIR / "movie_rating_model.joblib"

joblib.dump(final_model, MODEL_PATH)
print(f"Saved model to: {MODEL_PATH}")


## 12. Conclusions and limitations

- This task predicts a numeric movie rating, so it is a **regression** problem.
- The target rating is not imputed with zero; rows without a rating are excluded.
- Imputation, scaling, and one-hot encoding are fitted inside the pipeline after splitting.
- Director/actor average ratings calculated from the full dataset are intentionally excluded because they leak target information.
- Validation RMSE is used for model selection; the test set is reserved for one final evaluation.

**Limitations:** IMDb ratings and vote counts can change over time. Vote count may be strongly related to a movie's popularity and availability of ratings, and it may not be known at the time a prediction is needed. Random splitting does not measure future-year performance. A stronger next experiment would use a time-based split and evaluate models using only information available before the rating outcome.

**Future improvements:** add time-based evaluation, tune hyperparameters using training-only cross-validation, compare gradient-boosting models, and document the prediction use case and data limitations.
